In [3]:
import os
from pathlib import Path

import boto3
from botocore.exceptions import ClientError


def upload_directory(local_dir: Path, bucket: str, prefix: str = ""):
    """Recursively upload a local directory to S3."""
    local_dir = local_dir.resolve()
    if not local_dir.is_dir():
        raise ValueError(f"{local_dir} is not a directory")

    session = boto3.Session()  # uses your default AWS credentials
    s3_client = session.client("s3")

    prefix = prefix.strip("/")

    for root, _, files in os.walk(local_dir):
        for filename in files:
            local_path = Path(root) / filename
            rel_path = local_path.relative_to(local_dir)

            if prefix:
                s3_key = f"{prefix}/{rel_path.as_posix()}"
            else:
                s3_key = rel_path.as_posix()

            print(f"Uploading {local_path} -> s3://{bucket}/{s3_key}")
            try:
                s3_client.upload_file(str(local_path), bucket, s3_key)
            except ClientError as e:
                print(f"Failed to upload {local_path}: {e}")

In [4]:
from pathlib import Path
import os

# Configure these for your environment
BUCKET_NAME = "ishiki-ml-datasets"
S3_PREFIX = "mixed_dataset_v2"

# Local folder to upload: ami_code/mixed_dataset_v1 (run notebook from ami_code/)
LOCAL_DIR = Path("mixed_dataset_v2").resolve()

if not LOCAL_DIR.exists():
    raise FileNotFoundError(f"Local mixed_dataset_v2 folder not found at {LOCAL_DIR}. Run this notebook from ami_code/.")

print(f"Uploading from: {LOCAL_DIR}")
print(f"Uploading to: s3://{BUCKET_NAME}/{S3_PREFIX}")

# Perform upload
upload_directory(LOCAL_DIR, BUCKET_NAME, S3_PREFIX)

print(f"\nFinished uploading {LOCAL_DIR} to s3://{BUCKET_NAME}/{S3_PREFIX}")


Uploading from: /Users/manand/Documents/context_aware_modeling/ami_code/mixed_dataset_v2
Uploading to: s3://ishiki-ml-datasets/mixed_dataset_v2
Uploading /Users/manand/Documents/context_aware_modeling/ami_code/mixed_dataset_v2/.DS_Store -> s3://ishiki-ml-datasets/mixed_dataset_v2/.DS_Store
Uploading /Users/manand/Documents/context_aware_modeling/ami_code/mixed_dataset_v2/ami/.DS_Store -> s3://ishiki-ml-datasets/mixed_dataset_v2/ami/.DS_Store
Uploading /Users/manand/Documents/context_aware_modeling/ami_code/mixed_dataset_v2/ami/data_final/filtering_summary.json -> s3://ishiki-ml-datasets/mixed_dataset_v2/ami/data_final/filtering_summary.json
Uploading /Users/manand/Documents/context_aware_modeling/ami_code/mixed_dataset_v2/ami/data_final/stage4_filtered_samples.jsonl -> s3://ishiki-ml-datasets/mixed_dataset_v2/ami/data_final/stage4_filtered_samples.jsonl
Uploading /Users/manand/Documents/context_aware_modeling/ami_code/mixed_dataset_v2/spgi/.DS_Store -> s3://ishiki-ml-datasets/mixed_dat